# Fig. S3E, S3F: control vs KO cell-type composition

Cell-type composition of control vs 36-ASD-risk-KO cells per lineage (teal = control, gold = KO), horizontal 100%-stacked bars, and control vs KO proportions along pseudotime. KO cells are strongly over-represented among inhibitory progenitors/astrocytes but roughly balanced across excitatory types.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
import numpy as np, scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()
FIG = "Fig3/unassigned_panels"
TEAL, GOLD = "#298c8c", "#f1a226"  # control = teal, KO = gold (paper convention)
H5 = f"{DATA_ROOT}/3utr/organoids/knoblich/cellranger/aggregated_annotated_fix_filtered_normalized_dpt.h5ad"
ad = sc.read_h5ad(H5, backed="r")
o = ad.obs
_g = o["gRNA"].astype(str)
_i = o["induced"].astype(str)
keep = ~(
    (_i == "True") & ((_g == "nan") | (_g == "Control2"))
)  # confirmed-KO filter (matches s1c)
d = o[keep].copy()
d["cond"] = np.where(d["induced"].astype(str) == "True", "KO", "control")
d["celltype"] = d["celltype"].astype(str)
d["lineage"] = d["lineage"].astype(str)
ORD = {
    "Excitatory neurons": list(S.ORGANOID_EXC_ORDER),
    "Inhibitory neurons": list(S.ORGANOID_INH_ORDER),
}


# Per-lineage cell-type composition: fraction control (teal) vs KO (gold), horizontal 100%-stacked bars.
# The dashed line marks the lineage-wide control fraction; cell types left of it are KO-enriched.
def build_prop(L, stem, figsize, compact):
    sub = d[d.lineage == L]
    present = sub["celltype"].unique().tolist()
    order = [c for c in ORD[L] if c in present] + [
        c for c in present if c not in ORD[L]
    ]
    ct = (
        sub.groupby(["celltype", "cond"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(order)
    )
    tot = ct.sum(1)
    ctrl_frac = (ct["control"] / tot).values
    ko_frac = (ct["KO"] / tot).values
    overall = (sub.cond == "control").mean()
    yv = np.arange(len(order))[::-1]
    fig, ax = plt.subplots(figsize=figsize)
    ax.barh(yv, ctrl_frac, color=TEAL, edgecolor="white", linewidth=0.3, height=0.82)
    ax.barh(
        yv,
        ko_frac,
        left=ctrl_frac,
        color=GOLD,
        edgecolor="white",
        linewidth=0.3,
        height=0.82,
    )
    ax.axvline(overall, color="0.25", ls="--", lw=0.6, zorder=5)
    ax.set_yticks(yv)
    ax.set_yticklabels(order, fontsize=S.FONT_TICK - 0.5)
    ax.set_xlim(0, 1)
    ax.set_xticks([0, 0.5, 1.0])
    ax.set_xticklabels(["0", "50", "100"])
    ax.set_xlabel("% of cells", fontsize=S.FONT_LABEL)
    ax.tick_params(labelsize=S.FONT_TICK)
    ax.set_ylim(-0.6, len(order) - 0.4)
    ax.spines[["top", "right"]].set_visible(False)
    key = L.split()[0].lower()
    ax.set_title(
        L.split()[0][:3].upper(),
        loc="left",
        fontsize=S.FONT_ANNOT,
        color=S.paper_palette_lineage[key],
        fontweight="bold",
        pad=2,
    )
    ax.text(
        overall,
        len(order) - 0.35,
        f"{overall * 100:.0f}% ctrl",
        fontsize=S.FONT_LEGEND - 0.5,
        color="0.25",
        va="bottom",
        ha="center",
    )
    ax.legend(
        handles=[
            Patch(facecolor=TEAL, label="control"),
            Patch(facecolor=GOLD, label="KO"),
        ],
        fontsize=S.FONT_LEGEND,
        frameon=False,
        loc="center left",
        bbox_to_anchor=(1.0, 0.5),
        handlelength=1.0,
        handletextpad=0.4,
        labelspacing=0.4,
    )
    if compact:
        fig.subplots_adjust(left=0.24, right=0.80, top=0.86, bottom=0.24)
    else:
        fig.subplots_adjust(left=0.24, right=0.82, top=0.90, bottom=0.15)
    S.save_and_show(fig, f"cell_proportions_ctrl_vs_ko__{stem}", figure=FIG)


for _L, _s in [("Excitatory neurons", "exc"), ("Inhibitory neurons", "inh")]:
    build_prop(_L, f"{_s}_quarter_half", (S.HALF_W_IN, S.QUARTER_H_IN), False)
    build_prop(_L, f"{_s}_eighth_half", (S.HALF_W_IN, S.EIGHTH_H_IN), True)

## Control vs KO proportion over pseudotime (+ cell-type composition strip)


In [ ]:
# Cell proportions control vs KO over pseudotime (kernel-smoothed), with a cell-type composition
# strip along the pseudotime axis (as in the pseudotime-bin corrmat panels). Reuses `d` (filtered
# obs with cond) from above; teal = control, gold = KO; PAL = organoid cell-type colours.
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

PAL = S.paper_palette_organoid_celltypes
ORD_PT = {
    "Excitatory neurons": (list(S.ORGANOID_EXC_ORDER), "dpt_pseudotime_excitatory"),
    "Inhibitory neurons": (list(S.ORGANOID_INH_ORDER), "dpt_pseudotime_inhibitory"),
}


def _smooth_props(pt, cond, ctv, order, h=0.04, ng=200):
    grid = np.linspace(np.nanmin(pt), np.nanmax(pt), ng)
    ctrl = np.zeros(ng)
    comp = np.zeros((len(order), ng))
    isctrl = (cond == "control").astype(float)
    for j, gp in enumerate(grid):
        w = np.exp(-0.5 * ((pt - gp) / h) ** 2)
        sw = w.sum() + 1e-9
        ctrl[j] = (w * isctrl).sum() / sw
        for k, c in enumerate(order):
            comp[k, j] = (w * (ctv == c)).sum() / sw
    comp = comp / (comp.sum(0, keepdims=True) + 1e-9)
    return grid, ctrl, comp


def build_prop_pt(L, stem, figsize, compact):
    order, ptcol = ORD_PT[L]
    sub = d[(d.lineage == L) & np.isfinite(d[ptcol].astype(float))]
    pt = sub[ptcol].values.astype(float)
    cond = sub["cond"].values
    ctv = sub["celltype"].values
    order = [c for c in order if c in set(ctv)] + [
        c for c in np.unique(ctv) if c not in order
    ]
    grid, ctrl, comp = _smooth_props(pt, cond, ctv, order)
    overall = (cond == "control").mean()
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(
        2,
        1,
        height_ratios=[1, 0.12],
        hspace=0.06,
        left=0.13,
        right=(0.70 if not compact else 0.66),
        top=0.90,
        bottom=0.22,
    )
    axm = fig.add_subplot(gs[0])
    axs = fig.add_subplot(gs[1], sharex=axm)
    axm.fill_between(grid, 0, ctrl, color=TEAL, linewidth=0)
    axm.fill_between(grid, ctrl, 1, color=GOLD, linewidth=0)
    axm.axhline(overall, color="0.25", ls="--", lw=0.6)
    axm.set_xlim(grid[0], grid[-1])
    axm.set_ylim(0, 1)
    axm.set_yticks([0, 0.5, 1.0])
    axm.set_yticklabels(["0", "50", "100"])
    axm.set_ylabel("% of cells", fontsize=S.FONT_LABEL)
    axm.tick_params(labelsize=S.FONT_TICK, labelbottom=False)
    axm.spines[["top", "right"]].set_visible(False)
    key = L.split()[0].lower()
    axm.set_title(
        L.split()[0][:3].upper(),
        loc="left",
        fontsize=S.FONT_ANNOT,
        color=S.paper_palette_lineage[key],
        fontweight="bold",
        pad=2,
    )
    _mid = int(0.5 * len(grid))
    axm.text(
        grid[_mid],
        ctrl[_mid] * 0.5,
        "control",
        fontsize=S.FONT_LEGEND - 0.5,
        color="white",
        va="center",
        ha="center",
        fontweight="bold",
    )
    axm.text(
        grid[_mid],
        (ctrl[_mid] + 1) / 2,
        "KO",
        fontsize=S.FONT_LEGEND - 0.5,
        color="white",
        va="center",
        ha="center",
        fontweight="bold",
    )
    axm.text(
        grid[0],
        overall,
        f"{overall * 100:.0f}% ctrl",
        fontsize=S.FONT_LEGEND - 1,
        color="0.25",
        va="bottom",
        ha="left",
    )
    cum = np.zeros(len(grid))
    for k, c in enumerate(order):
        axs.fill_between(
            grid, cum, cum + comp[k], color=PAL.get(c, "#999999"), linewidth=0
        )
        cum = cum + comp[k]
    axs.set_xlim(grid[0], grid[-1])
    axs.set_ylim(0, 1)
    axs.set_yticks([])
    axs.set_xlabel("pseudotime", fontsize=S.FONT_LABEL)
    axs.tick_params(labelsize=S.FONT_TICK)
    if grid[-1] >= 0.9:
        axs.set_xticks([0, 0.5, 1.0])
    for sp in ["top", "right", "left"]:
        axs.spines[sp].set_visible(False)
    if not compact:
        axs.text(
            -0.02,
            0.5,
            "cell\ntype",
            transform=axs.transAxes,
            fontsize=S.FONT_LEGEND - 1,
            color="0.4",
            va="center",
            ha="right",
            linespacing=0.8,
        )
    handles = [Patch(facecolor=PAL.get(c, "#999999"), label=c) for c in order]
    fig.legend(
        handles=handles,
        loc="center left",
        bbox_to_anchor=((0.71 if not compact else 0.67), 0.5),
        ncol=1,
        frameon=False,
        fontsize=S.FONT_LEGEND - 0.5,
        handlelength=0.9,
        handletextpad=0.4,
        labelspacing=0.22,
    )
    S.save_and_show(fig, f"cell_proportions_pseudotime__{stem}", figure=FIG)


for _L, _s in [("Excitatory neurons", "exc"), ("Inhibitory neurons", "inh")]:
    build_prop_pt(_L, f"{_s}_quarter_half", (S.HALF_W_IN, S.QUARTER_H_IN), False)
    build_prop_pt(_L, f"{_s}_eighth_half", (S.HALF_W_IN, S.EIGHTH_H_IN), True)